<div dir="rtl" style="text-align:right; font-family:Tahoma,Arial,sans-serif; line-height:1.9; border-right:5px solid #20639b; padding:8px 18px;">
  <h1 style="color:#173f5f; margin-bottom:4px;">نوت‌بوک 18: تحلیل سری زمانی با Pandas</h1>
  <p><b>سطح:</b> متوسط &nbsp; | &nbsp; <b>نوع درس:</b> تکمیلی و کاربردی</p>
  <h3>هدف‌های یادگیری</h3>
  <ul><li>ساخت و استفاده از DatetimeIndex</li>
<li>انتخاب بازه‌های زمانی</li>
<li>تجمیع روزانه به هفتگی و ماهانه</li>
<li>محاسبهٔ میانگین متحرک و تغییر درصدی</li>
<li>شناسایی تاریخ‌های گمشده و مقایسهٔ دوره‌ها</li></ul>
</div>

<div dir="rtl" style="text-align:right; font-family:Tahoma,Arial,sans-serif; line-height:1.9;">
  <h2 style="color:#173f5f;">دادهٔ سری زمانی</h2>
  <p>در سری زمانی ترتیب تاریخ بخشی از معنای داده است. تبدیل صحیح ستون تاریخ، مرتب‌سازی و انتخاب فرکانس مناسب باید پیش از تحلیل انجام شود.</p>
</div>

In [1]:
# Build reproducible daily sales data
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
dates = pd.date_range("2026-01-01", periods=120, freq="D")
trend = np.linspace(200, 320, len(dates))
weekly_pattern = 35 * np.sin(2 * np.pi * np.arange(len(dates)) / 7)
noise = rng.normal(0, 18, len(dates))

sales = pd.DataFrame({
    "date": dates,
    "sales": np.maximum(trend + weekly_pattern + noise, 0).round(0),
}).set_index("date")

sales.head()


,sales
date,
2026-01-01,205.0
2026-01-02,210.0
2026-01-03,250.0
2026-01-04,235.0
2026-01-05,154.0


<div dir="rtl" style="text-align:right; font-family:Tahoma,Arial,sans-serif; line-height:1.9;">
  <h2 style="color:#173f5f;">DatetimeIndex و انتخاب بازه</h2>
  <p>وقتی تاریخ index باشد، می‌توان با رشتهٔ تاریخ روز، ماه یا بازه را انتخاب کرد.</p>
</div>

In [2]:
# Select a month and a date interval
january = sales.loc["2026-01"]
selected_week = sales.loc["2026-02-10":"2026-02-16"]

print("January rows:", len(january))
print("Selected week total:", selected_week["sales"].sum())


January rows: 31
Selected week total: 1742.0


<div dir="rtl" style="text-align:right; font-family:Tahoma,Arial,sans-serif; line-height:1.9;">
  <h2 style="color:#173f5f;">Resample</h2>
  <p><code>resample</code> فرکانس زمانی را تغییر می‌دهد. تابع تجمیع باید با معنای داده سازگار باشد؛ فروش را جمع می‌کنیم، اما دما معمولاً میانگین می‌شود.</p>
</div>

In [3]:
# Aggregate daily sales into weekly and monthly summaries
weekly_sales = sales["sales"].resample("W").sum()
monthly_summary = sales["sales"].resample("MS").agg(["sum", "mean", "max"])

print("Weekly totals:\n", weekly_sales.head())
monthly_summary.round(1)


Weekly totals:
 date
2026-01-04     900.0
2026-01-11    1387.0
2026-01-18    1517.0
2026-01-25    1560.0
2026-02-01    1654.0
Freq: W-SUN, Name: sales, dtype: float64


,sum,mean,max
date,,,
2026-01-01,6779.0,218.7,303.0
2026-02-01,6866.0,245.2,312.0
2026-03-01,8442.0,272.3,322.0
2026-04-01,8988.0,299.6,350.0


<div dir="rtl" style="text-align:right; font-family:Tahoma,Arial,sans-serif; line-height:1.9;">
  <h2 style="color:#173f5f;">پنجرهٔ متحرک</h2>
  <p>میانگین متحرک نوسان کوتاه‌مدت را هموار می‌کند. پارامتر <code>min_periods</code> مشخص می‌کند از چند مشاهده خروجی تولید شود.</p>
</div>

In [4]:
# Add rolling statistics without losing the original rows
sales["rolling_7d_mean"] = (
    sales["sales"]
    .rolling(window=7, min_periods=3)
    .mean()
)
sales["rolling_7d_std"] = (
    sales["sales"]
    .rolling(window=7, min_periods=3)
    .std()
)

sales.head(10).round(2)


,sales,rolling_7d_mean,rolling_7d_std
date,,,
2026-01-01,205.0,NaN,NaN
2026-01-02,210.0,NaN,NaN
2026-01-03,250.0,221.67,24.66
2026-01-04,235.0,225.00,21.21
2026-01-05,154.0,210.80,36.68
2026-01-06,147.0,200.17,41.89
2026-01-07,181.0,197.43,38.92
2026-01-08,201.0,196.86,38.82
2026-01-09,235.0,200.43,41.30


<div dir="rtl" style="text-align:right; font-family:Tahoma,Arial,sans-serif; line-height:1.9;">
  <h2 style="color:#173f5f;">تغییر درصدی و مقایسه با گذشته</h2>
  <p><code>pct_change</code> رشد نسبت به دورهٔ قبل را محاسبه می‌کند. با <code>shift</code> نیز مقدارهای گذشته را کنار دادهٔ فعلی قرار می‌دهیم.</p>
</div>

In [5]:
# Compare each day with the previous day and week
sales["previous_day"] = sales["sales"].shift(1)
sales["daily_change_pct"] = sales["sales"].pct_change() * 100
sales["change_vs_last_week"] = sales["sales"] - sales["sales"].shift(7)

sales[[
    "sales",
    "previous_day",
    "daily_change_pct",
    "change_vs_last_week",
]].head(10).round(2)


,sales,previous_day,daily_change_pct,change_vs_last_week
date,,,,
2026-01-01,205.0,NaN,NaN,NaN
2026-01-02,210.0,205.0,2.44,NaN
2026-01-03,250.0,210.0,19.05,NaN
2026-01-04,235.0,250.0,-6.00,NaN
2026-01-05,154.0,235.0,-34.47,NaN
2026-01-06,147.0,154.0,-4.55,NaN
2026-01-07,181.0,147.0,23.13,NaN
2026-01-08,201.0,181.0,11.05,-4.0
2026-01-09,235.0,201.0,16.92,25.0


<div dir="rtl" style="text-align:right; font-family:Tahoma,Arial,sans-serif; line-height:1.9;">
  <h2 style="color:#173f5f;">تاریخ گمشده</h2>
  <p>نبودن ردیف با مقدار NaN متفاوت است. ابتدا تقویم کامل را با <code>reindex</code> می‌سازیم و سپس دربارهٔ روش پرکردن تصمیم می‌گیریم.</p>
</div>

In [6]:
# Detect missing calendar dates and interpolate values
incomplete = sales.drop(index=sales.index[[12, 27, 45]])[["sales"]]
full_index = pd.date_range(incomplete.index.min(), incomplete.index.max(), freq="D")
restored = incomplete.reindex(full_index)

missing_dates = restored.index[restored["sales"].isna()]
restored["sales_interpolated"] = restored["sales"].interpolate(method="time")

print("Missing dates:", missing_dates.strftime("%Y-%m-%d").tolist())
print("Remaining gaps:", restored["sales_interpolated"].isna().sum())


Missing dates: ['2026-01-13', '2026-01-28', '2026-02-15']
Remaining gaps: 0


<div dir="rtl" style="text-align:right; font-family:Tahoma,Arial,sans-serif; line-height:1.9;">
  <h2 style="color:#173f5f;">گروه‌بندی تقویمی</h2>
  <p>ویژگی‌هایی مثل روز هفته، ماه و فصل به مقایسهٔ الگوهای تقویمی کمک می‌کنند.</p>
</div>

In [7]:
# Compare average sales by weekday
calendar_sales = sales.assign(
    weekday=sales.index.day_name(),
    weekday_number=sales.index.dayofweek,
)
weekday_summary = (
    calendar_sales.groupby(["weekday_number", "weekday"])
    .agg(
        average_sales=("sales", "mean"),
        total_sales=("sales", "sum"),
    )
    .reset_index()
    .sort_values("weekday_number")
)

weekday_summary.round(1)


,weekday_number,weekday,average_sales,total_sales
0,0,Monday,239.7,4075.0
1,1,Tuesday,227.5,3868.0
2,2,Wednesday,237.6,4040.0
3,3,Thursday,258.1,4645.0
4,4,Friday,283.5,4819.0
5,5,Saturday,292.2,4968.0
6,6,Sunday,274.1,4660.0


<div dir="rtl" style="text-align:right; font-family:Tahoma,Arial,sans-serif; line-height:1.9;">
  <h2 style="color:#173f5f;">تمرین</h2>
  <div style="background:#fff7df; border:1px solid #f0c36d; padding:12px; border-radius:8px;">تاریخی را پیدا کنید که میانگین متحرک هفت‌روزه در آن بیشترین مقدار را داشته است و فروش همان روز، میانگین متحرک و اختلافشان را گزارش دهید.</div>
</div>

<div dir="rtl" style="text-align:right; font-family:Tahoma,Arial,sans-serif; line-height:1.9;">
  <h2 style="color:#173f5f;">پاسخ پیشنهادی</h2>
  <p>ابتدا پاسخ خودتان را بنویسید و سپس با پاسخ پیشنهادی مقایسه کنید.</p>
</div>

In [8]:
# Find the peak seven-day moving average
peak_date = sales["rolling_7d_mean"].idxmax()
peak_row = sales.loc[peak_date]

peak_report = {
    "date": peak_date.strftime("%Y-%m-%d"),
    "sales": peak_row["sales"],
    "rolling_7d_mean": round(peak_row["rolling_7d_mean"], 2),
    "difference": round(
        peak_row["sales"] - peak_row["rolling_7d_mean"],
        2,
    ),
}
print(peak_report)


{'date': '2026-04-30', 'sales': np.float64(316.0), 'rolling_7d_mean': np.float64(316.14), 'difference': np.float64(-0.14)}


<div dir="rtl" style="text-align:right; font-family:Tahoma,Arial,sans-serif; line-height:1.9;">
  <h2 style="color:#173f5f;">جمع‌بندی</h2>
  <p>در سری زمانی ابتدا تقویم را کامل و مرتب کنید، سپس فرکانس، تجمیع، پنجرهٔ متحرک و مقایسه با دورهٔ گذشته را متناسب با سؤال تحلیل انتخاب کنید.</p>
</div>